# CS2 EXP-5 — HEFT (Hierarchical Efficient Fine-Tuning: LoRA + ReFT)


## 1. Dependencies


In [ ]:
import sys
import subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "tf-keras",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "peft<0.14.0",
    "pyreft",  # Required for HEFT (Hill, 2025)
    "accelerate",
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
], check=True)
print("Dependencies installed successfully for CUDA 12.1 driver!")

Dependencies installed successfully for CUDA 12.1 driver!


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")

Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 1.5 Settings


In [3]:
import os
from pathlib import Path
REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"
WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"
DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"
SPLIT_ID = "cs1_project_holdout20_innercv_v1"
NORMALIZED_PARQUET = (
    PROCESSED_DIR
    / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"
)
OUTER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "outer_holdout"
    / "cs1_outer_project_holdout_manifest.parquet"
)
INNER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "inner_cv"
    / "cs1_project_grouped_5fold_manifest.parquet"
)
EXP5_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "case_study_2"
    / "exp5_codeberta_heft_v1"
)
EXP5_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"
os.environ["HF_TOKEN"] = "secret"
RANK_GRID = (8, 16, 32)
EPOCHS = 4
SEARCH_EPOCHS = 2
SEARCH_N_SPLITS = 5
TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 8
STORAGE_CAP_GB = 60
RUN_SMOKE_TEST = True
RUN_NESTED_OFFICIAL = True
RUN_CANONICAL_RETRAIN = True
RUN_HOLDOUT_EVAL = True
print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
print(f"Device: {DEVICE}")

Settings loaded.
Workspace: /workspace
Repository: /workspace/DiverseVul--IS-Project
Data root: /workspace/IntelligentSystemProject/VulnerabilityDetectionData
Output root: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs
Hugging Face cache: /workspace/IntelligentSystemProject/hf_cache
Device: cuda:0


## 2. Clone the repository


In [4]:
import urllib.request
import zipfile
import shutil
from pathlib import Path
if REPO_ROOT.exists():
    print(f"Removing existing repository at {REPO_ROOT}...")
    shutil.rmtree(REPO_ROOT)
print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")
clean_url = REPO_URL.removesuffix(".git")
zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
zip_path = Path.cwd() / "repo_temp.zip"
urllib.request.urlretrieve(zip_url, zip_path)
print("Extracting files...")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(Path.cwd())
repo_name = clean_url.split("/")[-1]
extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
if extracted_folder.exists():
    extracted_folder.rename(REPO_ROOT)
zip_path.unlink()
print("Repository setup complete!")

Removing existing repository at /workspace/DiverseVul--IS-Project...
Extracting files...
Repository setup complete!


## 3. Verify GPU, RAM, and storage budget


In [5]:
import shutil
import psutil
import torch
if not torch.cuda.is_available():
    raise RuntimeError("EXP-5 requires a GPU runtime for HEFT fine-tuning.")
assert DEVICE == "cuda:0"
print("CUDA device:", torch.cuda.get_device_name(0))
print("bfloat16 supported:", torch.cuda.is_bf16_supported())
print("Total VRAM: %.2f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print("CPU cores:", psutil.cpu_count(logical=True))
print("System RAM: %.1f GB" % (psutil.virtual_memory().total / 1e9))
total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage already exceeds the {STORAGE_CAP_GB} GB storage cap for this machine.")

CUDA device: NVIDIA A100-SXM4-80GB
bfloat16 supported: True
Total VRAM: 84.99 GB
CPU cores: 112
System RAM: 2164.0 GB
Disk usage at /workspace: 5129.5 GB used / 5714.2 GB total (296.6 GB free)


## 4. Data availability check


In [6]:
required_data_files = {
    "normalized parquet": NORMALIZED_PARQUET,
    "outer holdout manifest": OUTER_MANIFEST_PATH,
    "inner CV manifest": INNER_MANIFEST_PATH,
}
missing = {name: path for name, path in required_data_files.items() if not path.is_file()}
if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by the Case Study 1 pipeline (normalization_v3.py + "
        "split_manifest.py) and were previously synced through Google Drive.\n"
        f"Keep an eye on the {STORAGE_CAP_GB} GB storage cap."
    )
    raise FileNotFoundError("Required processed data/manifests are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")

Found normalized parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet (210.7 MB)
Found outer holdout manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet (1.4 MB)
Found inner CV manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet (1.2 MB)


## 5. Import project modules



In [ ]:
import sys
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1"):
        del sys.modules[mod_name]
from case_study_2.data_loader import create_dataloader, get_class_weights
from case_study_2.models import (
    configure_huggingface_cache,
    load_code_tokenizer,
    DEFAULT_CODE_MODEL,
    DEFAULT_CODE_TOKENIZER,
    count_trainable_parameters,
    CodeSequenceClassifier,
    infer_lora_target_modules,
)
from case_study_2.exp5.exp5_heft import train_heft_model, train_heft_model_safe
from case_study_2.exp5.exp5_nested_pipeline import (
    Exp5Config,
    run_exp5_nested_rank,
    run_exp5_canonical_retrain,
    run_exp5_holdout_evaluation,
)
from case_study_1.confidence_intervals import paired_bootstrap_metric_ci, format_paired_ci_report

## 6. Load dataset and frozen manifests



In [14]:
import pandas as pd
full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)
print("full_df rows:", len(full_df))
print("outer_manifest_df rows:", len(outer_manifest_df))
print("inner_manifest_df rows:", len(inner_manifest_df))

## 7. Build development and holdout frames



In [15]:
required_columns = {"source_row_id", "normalized_code", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")
full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())
if dev_ids != inner_ids:
    raise RuntimeError("Development partition and inner manifest coverage do not match.")
if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and holdout partitions overlap.")
development_frame = full_indexed.loc[full_indexed["source_row_id"].isin(dev_ids)].reset_index(drop=True)
holdout_frame = full_indexed.loc[full_indexed["source_row_id"].isin(holdout_ids)].reset_index(drop=True)
print("development_frame rows:", len(development_frame))
print("holdout_frame rows:", len(holdout_frame))

## 8. Configuration object



In [16]:
exp5_config = Exp5Config(
    hf_cache_dir=HF_CACHE_DIR,
    rank_grid=RANK_GRID,
    epochs=EPOCHS,
    search_epochs=SEARCH_EPOCHS,
    search_n_splits=SEARCH_N_SPLITS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    eval_batch_size=EVAL_BATCH_SIZE,
)
print(exp5_config)

## 9. Sanity-check target modules



In [17]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)
_probe_model = CodeSequenceClassifier(model_name=DEFAULT_CODE_MODEL, freeze_backbone=False, dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred target_modules:", _target_modules)
del _probe_model
torch.cuda.empty_cache()

## 10. Smoke test on a small subsample



In [18]:
import time
from sklearn.metrics import average_precision_score
if RUN_SMOKE_TEST:
    sample_df = development_frame.sample(n=min(2000, len(development_frame)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)
    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")
    t0 = time.time()
    smoke_scores, smoke_model = train_heft_model_safe(
        smoke_train, smoke_val, _tok_check, rank=RANK_GRID[0], epochs=1,
        batch_size=exp5_config.train_batch_size, grad_accum_steps=exp5_config.grad_accum_steps,
        eval_batch_size=exp5_config.eval_batch_size, num_workers=exp5_config.num_workers,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=exp5_config.code_column,
        max_length=exp5_config.max_length,
    )
    smoke_prauc = float(average_precision_score(smoke_val[exp5_config.label_column].values, smoke_scores))
    print(f"[smoke] PR-AUC={smoke_prauc:.4f} | elapsed={(time.time()-t0)/60:.1f} min")
    del smoke_model
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")

## 11. Official nested HEFT rank search



In [19]:
nested_results = None
if RUN_NESTED_OFFICIAL:
    nested_results = run_exp5_nested_rank(
        development_frame=development_frame,
        development_manifest=inner_manifest_df,
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
        resume=True,
    )
    selected_ranks = nested_results["selected_rank"]["selected_rank"].tolist()
    best_rank = int(nested_results["selected_rank"]["selected_rank"].mode()[0])
    print(f"\nOuter fold selected ranks: {selected_ranks} -> Majority chosen rank for retrain: {best_rank}")

## 12. Canonical retrain and frozen outer holdout scoring



In [21]:
if RUN_CANONICAL_RETRAIN and nested_results is not None:
    global_selected_rank = int(nested_results["selected_rank"]["selected_rank"].mode()[0])
    retrain_results = run_exp5_canonical_retrain(
        development_frame=development_frame,
        tokenizer=nested_results["tokenizer"], 
        selected_rank=global_selected_rank,
        holdout_frame=holdout_frame,
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
    )
    print("Canonical model trained with rank =", global_selected_rank)
else:
    retrain_results = None
    print("RUN_CANONICAL_RETRAIN=False or no nested results; skipping.")

## 13. Frozen outer holdout evaluation with bootstrap confidence interval



In [22]:
if RUN_HOLDOUT_EVAL and retrain_results is not None:
    holdout_results = run_exp5_holdout_evaluation(
        holdout_predictions=retrain_results["holdout_predictions"],
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
    )
else:
    holdout_results = None
    print("RUN_HOLDOUT_EVAL=False or no canonical retrain results; skipping.")

## 14. Paired comparison against EXP-3 on the frozen holdout



In [23]:
EXP3_HOLDOUT_PREDICTIONS_PATH = OUTPUT_ROOT / "case_study_2" / "exp3_codeberta_linear_probe_v1" / "exp3_holdout_predictions.csv"
if RUN_HOLDOUT_EVAL and retrain_results is not None and EXP3_HOLDOUT_PREDICTIONS_PATH.exists():
    exp3_holdout = pd.read_csv(EXP3_HOLDOUT_PREDICTIONS_PATH)
    comparison = paired_bootstrap_metric_ci(
        predictions_a=retrain_results["holdout_predictions"],
        predictions_b=exp3_holdout,
        experiment_name_a="EXP-5 HEFT",
        experiment_name_b="EXP-3 linear probe",
        metric="average_precision_pr_auc",
    )
    print(format_paired_ci_report(comparison))
else:
    print("Run EXP-5 holdout evaluation and the EXP-3 notebook first.")

## 15. Cleanup



In [24]:
import gc
import shutil
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")
total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP5_OUTPUT_DIR}.")